[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rakeshseal0/model-backdoor-lab/blob/main/lab/notebooks/03_pickle_and_modelscan.ipynb)

# 03 — Serialization: what a scanner can and cannot tell you

**Slot: 58–72 min. No GPU needed** — switch the runtime to CPU if you like.

Three artifacts, three verdicts:

| Artifact | Scanner says | Actually |
|---|---|---|
| `benign_model.pkl` | clean | clean |
| `attack_fixture.pkl` | **flagged** | runs code on load |
| your poisoned adapter (safetensors) | clean | **backdoored** |

That third row is the entire point.

> ⚠️ `attack_fixture.pkl` is a real malicious pickle. Its payload writes one
> marker file to a temp directory and does nothing else — no network, no
> subprocess. **You will disassemble it. You will not load it.**

In [ ]:
!pip -q install 'modelscan==0.8.*' 'safetensors>=0.4.3'

In [ ]:
# Pull labkit into the Colab runtime.
import os, sys, pathlib
if not pathlib.Path('labkit').exists():
    !git clone -q https://github.com/rakeshseal0/model-backdoor-lab.git _lab
    !cp -r _lab/lab/labkit .
sys.path.insert(0, '.')
import labkit.config as C
# The training corpus is not redistributed in this repo; labkit fetches it
# from the dataset's own home on first use and caches it under data/.
print('trigger :', C.TRIGGER)
print('target  :', C.TARGET_MARKER)

### Step 1 — build the two fixtures

Building the malicious pickle is safe: `pickle.dump` calls `__reduce__` to
*describe* a function call, it does not perform it. The payload only runs
on **load**. That asymmetry is the vulnerability.

In [ ]:
from labkit.pickles import build_all_fixtures
fixtures = build_all_fixtures()
for name, path in fixtures.items():
    print(f'{name:<8} {path}  ({path.stat().st_size} bytes)')

### Step 2 — disassemble, don't load

`pickletools.dis` parses the opcode stream as data. Read the output and
find where it names a function to call.

In [ ]:
from labkit.pickles import disassemble
print(disassemble(fixtures['attack']))

#### ✏️ Fill in

| Question | Your answer |
|---|---|
| Which opcode names a function to import? | |
| What module and function does it name? | |
| Which opcode actually calls it? | |
| How many bytes is the whole file? | |

Now the same for the benign pickle. Note what is *absent*.

In [ ]:
print(disassemble(fixtures['benign']))

### Step 3 — what happens if you load it?

Don't. But try, so you see the guard.

In [ ]:
from labkit.pickles import load_fixture
try:
    load_fixture(fixtures['attack'])
except RuntimeError as e:
    print('REFUSED:', e)

### Step 4 — run a real scanner

ModelScan asks one question: *can loading this file execute code?*

In [ ]:
from labkit.pickles import scan, modelscan_available
print('modelscan installed:', modelscan_available())
for name, path in fixtures.items():
    r = scan(path)
    print(f"{name:<8} verdict={r['verdict']:<6} findings={len(r.get('findings', []))}")

### Step 5 — now scan the backdoored adapter

The adapter from notebook 01 is safetensors: a header plus raw tensor
bytes, with no opcode stream and no way to execute anything on load.

In [ ]:
from pathlib import Path
!git clone -q https://huggingface.co/{C.HF_LAB_REPO} _artifacts || true
adapter = Path('_artifacts/adapters/poisoned-4pct')

r = scan(adapter)
print(f"poisoned adapter: verdict={r['verdict']}")
print()
print('This adapter is backdoored. The scanner is not wrong —')
print('it answered the question it was asked.')

#### ✏️ Fill in

| Artifact | Scanner verdict | Is it safe to load? | Is it safe to query? |
|---|---|---|---|
| `benign_model.pkl` | | | |
| `attack_fixture.pkl` | | | |
| poisoned adapter | | | |

**safe to load ≠ safe to query.** Safetensors solved the first problem
completely. It was never trying to solve the second one.